# Parallel

In [6]:
import os 
import asyncio
import yfinance as yf
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
@function_tool
def get_fundamentals(ticker: str)->dict:
    """
    Fetches fundamental financial metrics for a stock.
    Args:
        ticker (str): The stock ticker symbol.
    """

    info = yf.Ticker(ticker).info
    fundamentals = {
        "ticker": ticker,
        "current_price": info.get("currentPrice"),
        "pe_ratio": info.get("trailingPE"),
        "forward_pe": info.get("forwardPE"),
        "revenue_growth": info.get("revenueGrowth"),
        "profit_margins": info.get("profitMargins"),
        "return_on_equity": info.get("returnOnEquity"),
        "earnings_growth": info.get("earningsGrowth"),
    }
    return fundamentals

@function_tool
def get_risk_metrics(ticker: str)->dict:
    """
    Fetches risk metrics for a stock.
    Args:
        ticker (str): The stock ticker symbol.
    """

    info = yf.Ticker(ticker).info
    risk_metrics = {
        "ticker": ticker,
        "beta": info.get("beta"),                          # volatility vs market
        "debt_to_equity": info.get("debtToEquity"),
        "current_ratio": info.get("currentRatio"),         # short-term liquidity
        "52_week_high": info.get("fiftyTwoWeekHigh"),
        "52_week_low": info.get("fiftyTwoWeekLow"),
        "short_ratio": info.get("shortRatio"),             # short interest
    }
    return risk_metrics

@function_tool
def get_market_sentiment(ticker: str)->dict:
    """
    Fetches market sentiment data for a stock.
    Args:
        ticker (str): The stock ticker symbol.
    """

    info = yf.Ticker(ticker).info
    sentiment = {
        "ticker": ticker,
        "analyst_recommendation": info.get("recommendationKey"),
        "target_price": info.get("targetMeanPrice"),
        "number_of_analysts": info.get("numberOfAnalystOpinions"),
        "institutional_ownership": info.get("heldPercentInstitutions"),
        "insider_ownership": info.get("heldPercentInsiders"),
    }
    return sentiment

In [3]:
fundamentals_agent = Agent(
    name="FundamentalsAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a fundamental analysis specialist.
    Always call get_fundamentals first

    Analyze: valuation, growth trajectory, profitability.
    Output exactly 3 bullet points. Be specific with numbers and metrics.
    End with: FUNDAMENTAL_SCORE: [1-10]    
    """,
    tools=[get_fundamentals]
)

risk_agent = Agent(
    name="RiskAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a risk assessment specialist.
    Always call get_risk_metrics first
    Analyze: volatility, debt levels, liquidity,short interest.
    Output exactly 3 bullet points. Be specific with numbers and metrics.
    End with: RISK_SCORE: [1-10]
    """,
    tools=[get_risk_metrics]
)

sentiment_agent = Agent(
    name="SentimentAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a market sentiment specialist.
    Always call get_market_sentiment first
    Analyze: analyst consensus, institutional confidence, insider activity.
    Output exactly 3 bullet points. Be specific with numbers and metrics.
    End with: SENTIMENT_SCORE: [1-10]
    """,
    tools=[get_market_sentiment]
)

In [4]:
import time

ticker = "NVDA"
query = f"Analyze {ticker} stock"

start = time.time()

# asyncio.gather runs all three simultaneously
fundamentals_result, risk_result, sentiment_result = await asyncio.gather(
    Runner.run(fundamentals_agent, query),
    Runner.run(risk_agent, query),
    Runner.run(sentiment_agent, query)
)

elapsed = time.time() - start

print(f"All three agents completed in {elapsed:.2f} seconds\n")
print("=== FUNDAMENTALS ===")
print(fundamentals_result.final_output)
print("\n=== RISK ===")
print(risk_result.final_output)
print("\n=== SENTIMENT ===")
print(sentiment_result.final_output)

All three agents completed in 10.04 seconds

=== FUNDAMENTALS ===
- **Valuation**: NVIDIA (NVDA) is currently trading at a P/E ratio of 31.95, which reflects a premium valuation compared to its forward P/E of 16.45, suggesting expectations of robust future growth.
  
- **Growth Trajectory**: Revenue growth stands at 85.2% year-over-year, and the earnings growth rate is an impressive 214.5%, indicating strong momentum in the company's performance and market opportunities.

- **Profitability**: NVIDIA boasts a high profit margin of 62.97% and an exceptional return on equity (ROE) of 114.29%, highlighting its efficiency in converting sales into profits and providing high returns to shareholders.

FUNDAMENTAL_SCORE: 9

=== RISK ===
- **Volatility**: NVDA has a beta of 2.202, indicating it is significantly more volatile than the market, which presents higher risks for investors.
  
- **Debt Levels**: The debt-to-equity ratio stands at 6.555, suggesting a high level of leverage which might a

In [5]:
synthesizer_agent = Agent(
    name="SynthesizerAgent",
    model="gpt-4o-mini",
    instructions="""
    You are senior investment analyst.
    You receive research from three specialist agents and produce the final report.

    You will be given
    - Fundamentals analysis + score
    - Risk analysis + score
    - Sentiment analysis + score

    Your output structure (follow this EXACTLY):
    ## Investment Report: [TICKER]
    
    ### Key Findings
    - Fundamentals: [1 sentence summary + score]
    - Risk: [1 sentence summary + score]  
    - Sentiment: [1 sentence summary + score]
    
    ### Overall Score
    [Weighted average: Fundamentals 40% + Sentiment 30% + Risk 30% inverted]
    
    ### Verdict
    [STRONG BUY / BUY / HOLD / AVOID]
    
    ### Reasoning
    [2-3 sentences connecting the three analyses into one coherent argument]
    
    ### Key Risk to Watch
    [Single biggest risk that could invalidate this thesis]

    """)


In [7]:
combined_input = f"""
Fundamentals Analysis:
{fundamentals_result.final_output}
Risk Analysis:
{risk_result.final_output}
Sentiment Analysis:
{sentiment_result.final_output}

Produce the final investment report based on the above analyses.
""" 

final_report = await Runner.run(synthesizer_agent, combined_input)


In [8]:
display(Markdown(final_report.final_output))

## Investment Report: NVDA

### Key Findings
- Fundamentals: NVIDIA shows exceptional growth with a yearly revenue increase of 85.2% and an earnings growth rate of 214.5%, achieving a Fundamental Score of 9.
- Risk: The company exhibits high volatility (beta of 2.202) and significant leverage, reflected in a Risk Score of 7.
- Sentiment: Strong institutional support and a favorable analyst consensus with a Sentiment Score of 8.

### Overall Score
Weighted average: (9 * 0.40) + (8 * 0.30) + (7 * 0.30) = 8.4

### Verdict
STRONG BUY

### Reasoning
NVIDIA's robust fundamentals, characterized by explosive growth and impressive profitability metrics, provide a strong foundation for long-term value appreciation. Coupled with positive sentiment from analysts and institutional investors, which underlines confidence in ongoing performance, the potential outweighs the elevated risk posed by market volatility and high leverage. 

### Key Risk to Watch
The primary risk is the company's high debt-to-equity ratio, which could impact its financial stability in the event of a market downturn or reduced revenue growth.